In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import os
import torch
import analysis as a
from mintrans_clean import *

# ---------------------------------------------------------------------------
# Empirical Structure Factor
# ---------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

files = os.listdir('./skeletons_var_lanbda')

path_wd = [{ "wd": decay.split('wd')[-1 :][0].split('_')[-1:][0].split('.pth')[0], "path": decay }for decay in files]

empirical_structure_factor = []

cfg = TrainConfig()

for obj in path_wd:
    path = os.path.join("skeletons_var_lanbda", obj['path'])
    weight_decay = obj['wd']

    model = MinimalTransformer(
        vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
        num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
        hidden_mlp=cfg.hidden_mlp,
    ).to(device)

    skeletons = torch.load(path, map_location=device)

    L = a.collect_state_logits(model, n=cfg.p, seq_len=cfg.seq_len,
        device=device, batch_size=cfg.batch_size,
        eq_token=cfg.p)

    empirical_structure_factor.append(a.oz_sf(L)[0])

# The Lazy Math

'''
    Everything in the universe wants to be in the lowest possible state,
    system is unstable if the energy is high. Like the Bohr's atomic model.

    Intuition: Lets understand this in terms of superconductor.

    F = normal + ( penalty 1 ) + ( penalty 2 ) + ( penalty 3 )

    consider "normal" as normal eneergy as if the metal was not in a superconducting state.

    penalty 1 ( cost of creating a superconducting state )

    penalty 2 ( if the superconducing state changes sharply from one spot to another,
    it costs energy so the superconductor prefers to become smooth and uniform ) -> cost of
    changing from place to place.

    penalty 3 ( superconductor kicks the magnetic fild out )

    What about phase transition?

    

'''
